# 대화 메모리 (Chat Memory)

- LLM 호출은 기본적으로 이전 호출을 기억하지 않는 stateless 요청이다.
- 대화 맥락을 유지하려면 매 호출에 이전 메시지를 함께 전달해야 한다.
- 이 노트북에서는 LangChain Core의 메시지와 히스토리 인터페이스를 사용해 직접 관리한다.

In [1]:
from dotenv import load_dotenv
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    trim_messages,
)
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
MODEL_NAME = "gemini-3.6-flash"
# 이 모델은 sampling 기본값이 고정되어 있으므로 temperature를 전달하지 않는다.
llm = ChatGoogleGenerativeAI(model=MODEL_NAME)

---

## 메모리 없는 대화

첫 번째 호출의 메시지를 두 번째 호출에 전달하지 않으면 모델은 이전 내용을 알 수 없다.

In [2]:
response1 = llm.invoke([HumanMessage(content="내 이름은 철수야")])
print("응답1:", response1.content)

response2 = llm.invoke([HumanMessage(content="내 이름이 뭐였지?")])
print("응답2:", response2.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


응답1: [{'type': 'text', 'text': '안녕하세요, 철수님! 만나서 반갑습니다. 오늘 어떤 도움이 필요하신가요?', 'extras': {'signature': 'EpwOCpkOARFNMg+/VEZnqB5bztMSTqDeb5n7AwSH34JZY2WNjvAAHpXpf7KcudwGs6zbd0yzXmW/JPgaTBhh8TnUUtSH9tqY8MsWzPV1JO9IF4KZ4ygZpm4N0MJwjnaEItm1Fh4VpJGgojKcqzq9NeroGCYGjkuWPhdUJFGTNBLnYhA8eTwGVT1c1SiRDPSf1kzL0TJhnkmALtBXEJ0TNvvT9TYZZjxHkYFOcUZ30dNc3drzHNGrp+1JuXLnKLHDE0VIqQc25f0pWz7DtRniI2aRlPzBD8kiPtxMg2yo7JmCnlDGUKcHrFn8zM4Icm6vgpVGHrmKeZSKNkBpwg4DUmGXoiE+mBKcMfexMFR7gikZ3nx8FnVOHY0ciR+riNs60wtzk9ZXk/mH/+5hUbTdpGTqY1sOutOPeIH+B7GTvRBSxPKtvYoQyf9lGAT+T4egqtBfQLAVilprhJ0W8u3ig6HC3P7lXnjbPbwZnIooQ1BaxkpkJoATsZWpo55uU0dDuDU+EUdyoPJR12Z95JRp2WOnFBriujQUDn7dmLSRvfdRkEf5J2j+wvmeShgCtgvVq9dCEU0YCocKPNrBwl0WQs/MX6f7TF+M39VZRrirafGIOoMkkjOvQULZO4eiVJOyu0M4pKfED/8rsLZ4zDZLp3O7568Zb+I38EnQMpjFbQynFutrFqWx6cLfTrd9sTIJaD0/brhPqaBgrC0ntmETAkGjxbKZhz+yHAINVMIbuVit947nzFIgAHoiR8qWjoiD+Wjzc+dVtprePfjKID/wUsE1uQQo4WNWIn5yuHkBCxGmhkLwQ8jW2Is+VhpNhPHhnPU0mQhEpTIABlCSsn2O/VufKetNOkM9nJOM77Uo0o9zzqwMWlbrYCHAD4QbvkuIjpPZ

## 수동으로 히스토리 전달하기

이전 메시지를 리스트에 누적하여 함께 전달하면 맥락이 유지된다.

In [3]:
messages = [
    SystemMessage(content="너는 친절한 상담사야."),
    HumanMessage(content="내 이름은 철수야"),
]
response1 = llm.invoke(messages)

messages.extend([response1, HumanMessage(content="내 이름이 뭐였지?")])
response2 = llm.invoke(messages)
print(response2.content)

[{'type': 'text', 'text': '방금 저에게 멋지게 소개해 주셨잖아요, **철수**님이요! 😊\n\n제가 철수님 이름 정확히 기억하고 있으니까 걱정 마세요. 혹시 저를 살짝 시험해 보신 건가요? ㅎㅎ\n\n언제든 철수님의 이야기를 들을 준비가 되어 있답니다. 편하게 이야기 나누어요!', 'extras': {'signature': 'EpsLCpgLARFNMg+ttaOf7uF4UmZs6ujGeDZCqwYDlTBw5Trpuzchl5l5RZr5PdlSLvX3fNdRzxHXnMqwfmG7ZdAq3fu0eCqjjPFDsV2ECiFgroo7fBf7AsgSvsDjj23TCUwgKISUpEZrPaXL7JYXvAWNSN6LsPf8nRKTorHrX81Kv/igHILbtOky/WO+7KLoSL3aRvf9sj3uEVMr+axC5SBZwT85o+SoYLI09scKrxxZ1LEOF1sJO8AhWcpIw+Rn/EsA+XZcR25KGR/kgyP6U0KcKd+MElS3nrbSLuBdCa1uxdnMpOHQoXJcJZio+wxsIHh0SUPjd8R2bWgMWolQC05F0DQSZ6R6ax4le6W25gM81YEx0LBJPs551onA9Mrv3jVbUjn6G0ugOMc6bMePgt+84GVY56S/6b8G9PEv5nxEd06R0pM7FFpgSUk7E+sFJ5hgl094YssODLRVDs5uVw3zCUSRcC8ZgX4bSX0agNIYAyZgYbBCMG6RjPzFpgYVjO21It6Ot2MahMyk1tIbwhEhDTnKPXXUxZ/J8vZIGlnjiqd2PLrGMn77tjwhT6EL9byYj+ihgE2Y/cTsnDFP9S8iY8amHYWJ5ZyONMzx0LKn1W5DIpYheZPy4Uig+L92qzvhA9xldCKi0+o0A8SHLu4DgbfaEXvrNhY8kIchoM3dpZ5ueFvXOKwss4ek4dEuNEfhb5ZOVnYYKsv6evZlyjQkw2oDNQYT+SpzRlSuq6MhfTyqqAIAJXX/1yOne8REQmNuXoAVL5XXfOJY1Ne91XhdaIfInc9fKCIp0

---

## InMemoryChatMessageHistory로 관리하기

`InMemoryChatMessageHistory`는 메시지 추가와 조회를 위한 LangChain Core의 기본 인메모리 구현이다. 세션 ID별로 객체를 나누면 여러 대화를 독립적으로 관리할 수 있다.

In [4]:
history_store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

def chat(session_id: str, user_input: str) -> str:
    history = get_session_history(session_id)
    user_message = HumanMessage(content=user_input)
    response = llm.invoke([
        SystemMessage(content="너는 친절한 상담사야."),
        *history.messages,
        user_message,
    ])
    history.add_messages([user_message, response])
    return response.content

In [5]:
print(chat("user-1", "내 이름은 철수야"))
print(chat("user-1", "내 이름이 뭐였지?"))

# 다른 세션에는 user-1의 기록이 없다.
print(chat("user-2", "내 이름이 뭐야?"))

[{'type': 'text', 'text': '반갑습니다, 철수님! 😊 \n저는 철수님의 이야기에 귀를 기울여 줄 친절한 상담사예요.\n\n오늘 어떤 하루를 보내셨나요? 혹시 나누고 싶은 이야기나 고민이 있으시다면 편하게 말씀해 주세요. 제가 항상 따뜻하게 들어드릴게요!', 'extras': {'signature': 'ErYMCrMMARFNMg9rRs3U9fmKNNw49TpT/g4pF28bBp3T4kfscSts2R/oMFrqBWmBtw1if/8f07FiS/dFEeUeHNLYypsaGaOTw8bFHOjBW1+wq77eQHNI5sL/zKDU8PfAQnl+PdSas3NIDrNo/5VSQe9sdV6We3OJBQYz3iYm+kqBiTKdljMr1Y3uLiZDqpzLRbX4PkuWZOEIb6YhLUcLTA4lKG4La35iIc+vNp3H33z9jfyOooWVajV9VjrBMVUfF+0JpXcEwPQP4ojNkZQhCT1Pw7EDKQH/9bbzKoV+eQHjaqshTXYtcWIGIvatSEkK0lglU5WmpgWVjkhiDQJegQMRS68BLeCv0JH+ue+N/psb2FHeeYRZtAgeIwsNCrXYcD1vAthcUZU8x1ucFqbLtVd+IyzM9jd6j+AlHDmobaX4FVE437xG3Ij2RbCiDPCKpuibhssG+1YxX/QTxyNZj4gX2CI8aQIYTz2bYhzx3+OqtohcBxKwkmfrqR+ZZ/DDeli3v4RN4EaUZEMmvg0deR9Xx0BeYNebPuffVnfGR4Z0MIbxvMNQ0np3/zaJu3mpdc3GfR31zWJZ5ZqJS190DT+FKU46x8JhKqkioek+70sZQy681CvwOEVI3ahewlQD5SRUYIvQowuKaHJVYm1cRfDg5L0qiGHijkTL7H/7f50efEe1HwavW34STDi82NHYOoPPjTbRaTXQtbvMysqs7YW9V0IWJlq7LAQ9D0VvTWjrO7jq5sKUpAsJgM7atsrpTr+LS4gTuaHM1DHWhRmI8rnqafsQ6BUpXEWW5a+7CH3aGW4CFhUS

In [6]:
for message in get_session_history("user-1").messages:
    print(f"[{message.type}] {message.content}")

[human] 내 이름은 철수야
[ai] [{'type': 'text', 'text': '반갑습니다, 철수님! 😊 \n저는 철수님의 이야기에 귀를 기울여 줄 친절한 상담사예요.\n\n오늘 어떤 하루를 보내셨나요? 혹시 나누고 싶은 이야기나 고민이 있으시다면 편하게 말씀해 주세요. 제가 항상 따뜻하게 들어드릴게요!', 'extras': {'signature': 'ErYMCrMMARFNMg9rRs3U9fmKNNw49TpT/g4pF28bBp3T4kfscSts2R/oMFrqBWmBtw1if/8f07FiS/dFEeUeHNLYypsaGaOTw8bFHOjBW1+wq77eQHNI5sL/zKDU8PfAQnl+PdSas3NIDrNo/5VSQe9sdV6We3OJBQYz3iYm+kqBiTKdljMr1Y3uLiZDqpzLRbX4PkuWZOEIb6YhLUcLTA4lKG4La35iIc+vNp3H33z9jfyOooWVajV9VjrBMVUfF+0JpXcEwPQP4ojNkZQhCT1Pw7EDKQH/9bbzKoV+eQHjaqshTXYtcWIGIvatSEkK0lglU5WmpgWVjkhiDQJegQMRS68BLeCv0JH+ue+N/psb2FHeeYRZtAgeIwsNCrXYcD1vAthcUZU8x1ucFqbLtVd+IyzM9jd6j+AlHDmobaX4FVE437xG3Ij2RbCiDPCKpuibhssG+1YxX/QTxyNZj4gX2CI8aQIYTz2bYhzx3+OqtohcBxKwkmfrqR+ZZ/DDeli3v4RN4EaUZEMmvg0deR9Xx0BeYNebPuffVnfGR4Z0MIbxvMNQ0np3/zaJu3mpdc3GfR31zWJZ5ZqJS190DT+FKU46x8JhKqkioek+70sZQy681CvwOEVI3ahewlQD5SRUYIvQowuKaHJVYm1cRfDg5L0qiGHijkTL7H/7f50efEe1HwavW34STDi82NHYOoPPjTbRaTXQtbvMysqs7YW9V0IWJlq7LAQ9D0VvTWjrO7jq5sKUpAsJgM7atsrpTr+LS4gTuaHM1DHWhRmI8rnqafsQ6

### 인메모리 히스토리의 한계

`InMemoryChatMessageHistory`는 대화 히스토리의 기본 동작을 학습하기에 적합하지만, 프로세스가 종료되면 기록이 사라진다. 이후 LangGraph에서는 같은 흐름을 state와 checkpointer로 관리하며, 데이터베이스 기반 저장소를 연결해 대화를 영속적으로 유지할 수 있다.

---

### Context window 관리

DB에 원본 메시지를 저장하는 것과 모델에 보낼 메시지를 선택하는 것은 별개의 문제다. 대화가 길어지면 비용과 지연 시간이 증가하고 context window를 넘을 수 있다.

| 방법 | 장점 | 단점 |
|---|---|---|
| 메시지 트리밍 | 단순하고 추가 호출 비용이 없음 | 오래된 맥락 유실 |
| 대화 요약 | 오래된 핵심 맥락 압축 | 추가 호출 비용과 요약 오류 가능성 |
| 검색 기반 선택 | 관련 과거 정보만 선택 가능 | 검색·인덱싱 설계 필요 |

요약과 최근 메시지를 함께 유지하는 방식은 유용한 일반 패턴이지만, 서비스 특성과 평가 결과에 맞춰 선택해야 한다.

`trim_messages()`는 전체 히스토리에서 모델에 전달할 메시지만 선택한다. 원본 리스트나 `InMemoryChatMessageHistory`의 메시지를 삭제하지 않고, 조건에 맞게 선택된 새로운 메시지 목록을 반환한다. 따라서 전체 대화는 저장소에 보존하면서 매 호출에 필요한 최근 대화만 모델에 전달할 수 있다.

`trim_messages()`의 주요 옵션은 다음과 같다.

- `max_tokens`: 트리밍 결과에 허용할 최대 토큰 수를 지정한다. 실제로 남는 메시지 수는 각 메시지의 길이에 따라 달라진다
- `strategy`: 앞쪽 또는 뒤쪽 중 어느 메시지를 우선하여 남길지 결정한다
- `token_counter`: 메시지의 토큰 수를 계산할 모델이나 함수를 지정한다
- `include_system`: 첫 system 메시지를 트리밍 결과에 유지할지 결정한다
- `start_on`: 트리밍된 대화가 어떤 역할의 메시지부터 시작해야 하는지 지정한다
- `end_on`: 트리밍된 대화가 어떤 역할의 메시지에서 끝나야 하는지 지정한다

아래 예제에서는 최대 토큰 수를 80으로 제한하고 최근 대화를 우선하여 남긴다. system 메시지는 유지하며, 그다음 대화가 human 메시지부터 시작하도록 설정한다.

처리 흐름은 다음과 같다.

```text
전체 히스토리 저장 → 토큰 수 계산 → 최근 메시지 선택 → 대화 시작 역할 정리 → 모델에 전달
```

트리밍 결과에 오래된 메시지가 포함되지 않으면 모델은 그 내용을 알 수 없다. 중요한 사용자 정보까지 단순히 제거될 수 있으므로, 실제 서비스에서는 최근 메시지와 대화 요약 또는 검색한 과거 정보를 함께 전달하는 방법을 고려한다.

In [7]:
long_history = [
    SystemMessage(content="너는 친절한 상담사야."),
    HumanMessage(content="안녕하세요"),
    AIMessage(content="안녕하세요. 무엇을 도와드릴까요?"),
    HumanMessage(content="Python에 대해 알려줘"),
    AIMessage(content="Python은 범용 프로그래밍 언어입니다."),
    HumanMessage(content="FastAPI에 대해 알려줘"),
    AIMessage(content="FastAPI는 Python 웹 프레임워크입니다."),
    HumanMessage(content="내 이름은 철수야"),
    AIMessage(content="반가워요, 철수님."),
    HumanMessage(content="내 이름이 뭐였지?"),
]

trimmer = trim_messages(
    max_tokens=80,
    strategy="last",
    token_counter=llm,
    include_system=True,
    start_on="human",
)
trimmed = trimmer.invoke(long_history)
print("원본 메시지 수:", len(long_history))
print("트리밍 후 메시지 수:", len(trimmed))
for message in trimmed:
    print(f"[{message.type}] {message.content}")

원본 메시지 수: 10
트리밍 후 메시지 수: 8
[system] 너는 친절한 상담사야.
[human] Python에 대해 알려줘
[ai] Python은 범용 프로그래밍 언어입니다.
[human] FastAPI에 대해 알려줘
[ai] FastAPI는 Python 웹 프레임워크입니다.
[human] 내 이름은 철수야
[ai] 반가워요, 철수님.
[human] 내 이름이 뭐였지?


---

## 참고: Short-term memory와 long-term memory

두 메모리는 단순히 저장 기간이 아니라 기억을 사용하는 **범위(scope)** 로 구분한다.

| 구분 | Short-term memory | Long-term memory |
|---|---|---|
| 범위 | 현재 대화 세션 | 여러 대화 세션 |
| 저장 내용 | 현재 대화 메시지와 작업 맥락 | 사용자 선호, 프로필, 기억할 사실이나 경험 |
| 예시 | "앞에서 내 이름을 철수라고 말했어" | "이 사용자는 Python 백엔드 개발자다" |

이 노트북에서는 현재 대화의 히스토리인 short-term memory만 다룬다. 세션을 넘어 정보를 저장하고 필요한 기억을 찾는 long-term memory는 이후 과정에서 별도로 다룬다.

---

### 실습문제

**세션별 대화 메모리 만들기**

`InMemoryChatMessageHistory`를 사용하여 세션별로 대화 내용을 기억하는 함수를 만들어보자. 같은 세션에서는 이전 대화를 기억하고, 서로 다른 세션의 대화는 섞이지 않는지 확인한다.

In [ ]:
history_store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

def chat(session_id: str, user_input: str) -> str:
    history = get_session_history(session_id)
    user_message = HumanMessage(content=user_input)
    response = llm.invoke([
        SystemMessage(content="너는 친절한 상담사야."),
        *history.messages,
        user_message,
    ])
    history.add_messages([user_message, response])
    return response.content

In [11]:
history_store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

def chat(session_id: str, user_input: str) -> str:
    history = get_session_history(session_id)
    user_message = HumanMessage(content=user_input)
    response = llm.invoke([
        SystemMessage(content="너는 친절한 상담사야."),
        *history.messages,
        user_message,
    ])
    history.add_messages([user_message, response])
    return response.content

In [12]:
print(chat("user-1", "내 이름은 철수야"))
print(chat("user-1", "내 이름이 뭐였지?"))

# 다른 세션에는 user-1의 기록이 없다.
print(chat("user-2", "내 이름이 뭐야?"))

[{'type': 'text', 'text': '안녕하세요, 철수님! 만나서 정말 반가워요. 😊\n\n오늘 어떤 마음으로 찾아오셨나요? 마음속에 있는 이야기나 요즘 드는 생각들, 편하게 말씀해 주세요. 제가 따뜻하게 들어드릴게요.', 'extras': {'signature': 'EvkNCvYNARFNMg+1f+tGHxzHkqNXsemc1EPxwJxGDSABFm/VJMrQtwdI6slkDfJCXQvH/oFMuuD+9reXRCCdEqQf88UcH6ebGfapyF004x28lgIo3Dtjo6LNbdLC9vK9HztLx2vXGlvSwdfYSw/EB5b1Zu3b+KDdqz14CnV5pAMoNWyfnh1+PGebQE4FjxQsGI0u8E+NtRAXrR/MfqQMfwibz8S4OcxgG2TZKg17KCw8za63XfLHu6LB1SSkV7oCitKzQbixOQ+kOWwhPYYZaeqYQWl8rTjyS141vDG4H7p+/nDJlXegwlcpnSw+s8vK91YT7ErR9MkapFTBYc28YTbBjXL9HNCQnxwslm+lrEoIUZDiYG0QDIyR/zeW3tq/CLa3VpfVQk1myfStKKo6aPr0pqinCnAm9wlsWl/+NHQGT5o7RgyDgsz4/iSbh0debfdsavZB8F0vQy6jX+UD4h4bOjHgGq1yCdca2zCRtZGxtfA8cyrtyfmKK/fgpwQRPQUY8RzTvCXG57w5qeNqiL02UB1dKTghgDBie9c0ij20yZFABYaOA98aSS5OeWd4sVkOiTV5k3nRDM+XT839ZYPQ+KEVmUaKMTCEfxUe5z2WPPHoDHU4Pm9ZFW+N8J4h6V733wyFoSj5SjqXCslb9MLowUOoNsg82n9SPUiJvTyHUV1mjFGTxxP2djnqSyXml4Q5Sb5CR695fz7yQDtmfE8UGJzQXrdmYr3FhfcAalX+xtT6LDgqQfjJxamnv3hmPEvag9obkDq2Y+CXkurzp+fnbcInmvc+ykLUfUMwbvzmn9fQ+uuCOgf7oOLZdeTlT3ETS7vO9o2I

**최근 대화만 전달하기**

위에서 만든 세션별 대화 함수에 `trim_messages()`를 적용해보자. 전체 대화는 히스토리에 보존하면서 모델에는 최근 대화만 전달한다. 충분히 긴 대화를 실행한 뒤 전체 메시지 수와 모델에 전달된 메시지 수가 달라지는지 확인한다.